# MISRA C: zip in, fixes and report out (Google Colab)

Upload a zip of your C project. This notebook finds MISRA C:2012 findings, asks an AI model to fix
them, keeps a fix only if the checker's gates pass, and gives you a report plus a fixed copy of the project.

**Runtime:** use *Runtime > Change runtime type > T4 GPU* if you want the local model (option B) or training (part 2).

**Read before using:** the gates prove "still compiles, targeted findings gone, nothing worse", not that behaviour
is unchanged. This is not a MISRA compliance claim. Review every patch.

Run the cells top to bottom.

## 1. Install the tools and build the checker (about 5 minutes)

In [ ]:
!cat /etc/os-release | head -2
!apt-get update -qq && apt-get install -y -qq clang-14 llvm-14-dev libclang-14-dev libclang-cpp14-dev cmake ninja-build zstd > /dev/null && echo apt-ok

In [ ]:
# Your repository and branch. If the repo is private, add a GitHub token as a Colab secret named GITHUB_TOKEN
# (key icon in the left bar) and it will be used below.
REPO   = "arrunraj66/misra-checker"
BRANCH = "claude/stoic-tesla-a53igu"   # change to "main" after you merge

import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token+'@' if token else ''}github.com/{REPO}.git"
if not os.path.exists("/content/misra-checker"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "/content/misra-checker"], check=True)
%cd /content/misra-checker
!git log --oneline -1

In [ ]:
!cmake -S . -B build -G Ninja -DLLVM_DIR=/usr/lib/llvm-14/lib/cmake/llvm > /dev/null && cmake --build build 2>&1 | tail -2
!./build/misra-checker --list-rules | wc -l

In [ ]:
# Development licence (the build uses the development key). Replace for real use.
!printf 'development-key-do-not-ship' > /content/dev.key
!python3 scripts/issue_license.py --customer colab --expires 2027-12-31 --features convert --key-file /content/dev.key > /content/colab.lic
!ls -l /content/colab.lic

## 2. Choose the AI that proposes fixes
Run **one** of the two cells.

**Option A - Claude API.** Strongest. Your source code is sent to Anthropic's API, so only use code you may share.
Add `ANTHROPIC_API_KEY` as a Colab secret (key icon, left bar) and switch on notebook access.

**Option B - local model on the Colab GPU.** Code stays in this Colab session. Uses Ollama with
`qwen2.5-coder:7b`, or your own fine-tuned `.gguf` from Google Drive (set `GGUF_PATH`).

In [ ]:
# OPTION A: Claude
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
PROVIDER = "python3 /content/misra-checker/scripts/ai_providers/claude_provider.py"

In [ ]:
# OPTION B: local model with Ollama on the GPU
import os, subprocess, time
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.Popen("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True)
time.sleep(8)

GGUF_PATH = ""   # e.g. "/content/drive/MyDrive/misra-fixer-q4_k_m.gguf" for your fine-tuned model; empty = base model
if GGUF_PATH:
    open("/content/Modelfile", "w").write(f"FROM {GGUF_PATH}\nPARAMETER temperature 0\nPARAMETER num_ctx 16384\n")
    subprocess.run("ollama create misra-fixer -f /content/Modelfile", shell=True, check=True)
    os.environ["MISRA_AI_MODEL"] = "misra-fixer"
else:
    subprocess.run("ollama pull qwen2.5-coder:7b", shell=True, check=True)
    os.environ["MISRA_AI_MODEL"] = "qwen2.5-coder:7b"
os.environ["MISRA_AI_NUM_CTX"] = "16384"
PROVIDER = "python3 /content/misra-checker/scripts/ai_providers/ollama_provider.py"

## 3. Upload your project zip
Zip the folder that holds `main.c`, `uart.c`, `usb.c`, the headers and the bundled libraries.

In [ ]:
from google.colab import files
uploaded = files.upload()
ZIP = "/content/" + next(iter(uploaded))
open(ZIP, "wb").write(uploaded[next(iter(uploaded))])
print(ZIP)

## 4. Find violations, convert, write the report
Settings: add defines such as `STM32F4`, and `EXCLUDE` globs for vendor or library folders you do not want changed.
`MAX_FILES = 0` converts every file with findings (the full project). For a quick first try, set it to a small number like 3.
Replace `VERIFY_CMD` with a command that builds and tests your project, if you have one that works in Colab.

In [ ]:
DEFINES   = []              # e.g. ["STM32F4", "DEBUG=1"]
EXCLUDE   = []              # whole project. To skip folders: ["lib/*", "vendor/*"]
MAX_FILES = 0               # 0 = the full project (every file with findings)
VERIFY_CMD = "true"
STD = "c99"

import shlex
cmd = ["python3", "scripts/run_project_pipeline.py", ZIP, "--out", "/content/result",
       "--checker", "build/misra-checker", "--provider-cmd", PROVIDER,
       "--license", "/content/colab.lic", "--verify-cmd", VERIFY_CMD,
       "--std", STD, "--max-files", str(MAX_FILES)]
for d in DEFINES: cmd += ["-D", d]
for e in EXCLUDE: cmd += ["--exclude", e]
!{" ".join(shlex.quote(c) for c in cmd)}

In [ ]:
# Show the report here and download the results
import IPython, shutil
IPython.display.display(IPython.display.HTML(open("/content/result/report.html").read()))
shutil.make_archive("/content/misra_results", "zip", "/content/result", "converter")
from google.colab import files
for f in ["/content/result/report.html", "/content/result/findings.csv", "/content/result/fixed_project.zip", "/content/misra_results.zip"]:
    files.download(f)

## 5. (Optional) Train the model on what worked
Every accepted fix is a verified training example. Run section 4 on many files and projects first
(set `MAX_FILES = 0`), keep only the patches you would accept in review, then continue.
A few hundred pairs is a start; rules with few examples stay weak. Use the T4 GPU runtime.

In [ ]:
# Build the dataset from accepted runs (each converter/N folder is one run)
!python3 scripts/build_training_data.py /content/result/converter/* --dataset /content/train.jsonl --eval /content/eval.jsonl
!wc -l /content/train.jsonl /content/eval.jsonl

In [ ]:
!pip install -q unsloth
from unsloth import FastLanguageModel
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-Coder-7B-Instruct", max_seq_length=8192, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

def load(path):
    rows = [json.loads(l) for l in open(path)]
    return Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows])

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=load("/content/train.jsonl"),
    eval_dataset=load("/content/eval.jsonl"),
    args=SFTConfig(per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, logging_steps=5, eval_strategy="epoch",
        output_dir="out", dataset_text_field="text", max_seq_length=8192, fp16=True))
trainer.train()

In [ ]:
# Save the model to Google Drive as a .gguf (so it survives the session), then copy it to your PC
from google.colab import drive
drive.mount("/content/drive")
model.save_pretrained_gguf("/content/drive/MyDrive/misra-fixer", tok, quantization_method="q4_k_m")

Next, on your PC: put the `.gguf` in `./models/`, run `docker compose --profile llm up -d` and
`docker/import-model.sh models/<file>.gguf misra-fixer` (see `docs/ai-training-guide.md`, section 6).
To compare base and fine-tuned models, run section 4 on files that were **not** in training with each model
and keep the fine-tuned one only if more of its fixes pass the gates.

Unsloth's interface changes often. If a cell in section 5 fails, check its current docs.